In [1]:
import pandas as pd
import numpy as np

# 1. Define the file paths for the Training data
path_target = r"D:\MSC\Semester 1\IT5022 - Fundamentals of Machine Learning\Assignment\Assignment 1\ML_Assignment\Dataset\Train-1542865627584.csv"
path_beneficiary = r"D:\MSC\Semester 1\IT5022 - Fundamentals of Machine Learning\Assignment\Assignment 1\ML_Assignment\Dataset\Train_Beneficiarydata-1542865627584.csv"
path_inpatient = r"D:\MSC\Semester 1\IT5022 - Fundamentals of Machine Learning\Assignment\Assignment 1\ML_Assignment\Dataset\Train_Inpatientdata-1542865627584.csv"
path_outpatient = r"D:\MSC\Semester 1\IT5022 - Fundamentals of Machine Learning\Assignment\Assignment 1\ML_Assignment\Dataset\Train_Outpatientdata-1542865627584.csv"

# 2. Load the CSV files into Pandas DataFrames
print("Loading data...")
df_target = pd.read_csv(path_target)
df_ben = pd.read_csv(path_beneficiary)
df_in = pd.read_csv(path_inpatient)
df_out = pd.read_csv(path_outpatient)

# 3. Check the shapes (number of rows and columns) of each table
print("\n--- Data Shapes ---")
print(f"Target Labels (Providers): {df_target.shape}")
print(f"Beneficiary (Patients): {df_ben.shape}")
print(f"Inpatient Claims: {df_in.shape}")
print(f"Outpatient Claims: {df_out.shape}")

C:\Users\nipun.ruwanpathirana.SOFTLOGIC\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\nipun.ruwanpathirana.SOFTLOGIC\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Loading data...

--- Data Shapes ---
Target Labels (Providers): (5410, 2)
Beneficiary (Patients): (138556, 25)
Inpatient Claims: (40474, 30)
Outpatient Claims: (517737, 27)


In [2]:
# 1. Add a column to distinguish the type of claim
df_in['ClaimType'] = 'Inpatient'
df_out['ClaimType'] = 'Outpatient'

# 2. Combine Inpatient and Outpatient claims into one master claims table
# pd.concat stacks them on top of each other. 
df_all_claims = pd.concat([df_in, df_out], ignore_index=True)

# 3. Merge the Patient (Beneficiary) details into the combined claims table
# We use a 'left' join to keep all claims, attaching patient info based on 'BeneID'
df_merged = pd.merge(df_all_claims, df_ben, on='BeneID', how='left')

# 4. Check the results
print("--- After Merging ---")
print(f"Master Claims Table Shape: {df_merged.shape}")
print("\nFirst 5 columns to verify the merge:")
print(df_merged.head())

--- After Merging ---
Master Claims Table Shape: (558211, 55)

First 5 columns to verify the merge:
      BeneID   ClaimID ClaimStartDt  ClaimEndDt  Provider  \
0  BENE11001  CLM46614   2009-04-12  2009-04-18  PRV55912   
1  BENE11001  CLM66048   2009-08-31  2009-09-02  PRV55907   
2  BENE11001  CLM68358   2009-09-17  2009-09-20  PRV56046   
3  BENE11011  CLM38412   2009-02-14  2009-02-22  PRV52405   
4  BENE11014  CLM63689   2009-08-13  2009-08-30  PRV56614   

   InscClaimAmtReimbursed AttendingPhysician OperatingPhysician  \
0                   26000          PHY390922                NaN   
1                    5000          PHY318495          PHY318495   
2                    5000          PHY372395                NaN   
3                    5000          PHY369659          PHY392961   
4                   10000          PHY379376          PHY398258   

  OtherPhysician AdmissionDt  ... ChronicCond_Depression  \
0            NaN  2009-04-12  ...                      1   
1         

In [3]:
# 1. Convert date strings to actual DateTime objects
df_merged['ClaimStartDt'] = pd.to_datetime(df_merged['ClaimStartDt'])
df_merged['ClaimEndDt'] = pd.to_datetime(df_merged['ClaimEndDt'])

# 2. Calculate the duration of each claim (in days)
df_merged['ClaimDuration'] = (df_merged['ClaimEndDt'] - df_merged['ClaimStartDt']).dt.days

# 3. Aggregate data at the Provider level (Feature Engineering)
print("Aggregating features per Provider...")
provider_features = df_merged.groupby('Provider').agg(
    Total_Claims=('ClaimID', 'count'),
    Unique_Patients=('BeneID', 'nunique'),
    Total_Claim_Amt=('InscClaimAmtReimbursed', 'sum'),
    Avg_Claim_Amt=('InscClaimAmtReimbursed', 'mean'),
    Avg_Claim_Duration=('ClaimDuration', 'mean')
).reset_index()

# 4. Merge these aggregated features with our Target Labels (PotentialFraud)
df_final = pd.merge(provider_features, df_target, on='Provider', how='inner')

# 5. Check the final dataset ready for Machine Learning
print("\n--- Final Provider-Level Dataset ---")
print(f"Shape: {df_final.shape}")
print("\nFirst 5 rows:")
print(df_final.head())

Aggregating features per Provider...

--- Final Provider-Level Dataset ---
Shape: (5410, 7)

First 5 rows:
   Provider  Total_Claims  Unique_Patients  Total_Claim_Amt  Avg_Claim_Amt  \
0  PRV51001            25               24           104640    4185.600000   
1  PRV51003           132              117           605670    4588.409091   
2  PRV51004           149              138            52170     350.134228   
3  PRV51005          1165              495           280910     241.124464   
4  PRV51007            72               58            33710     468.194444   

   Avg_Claim_Duration PotentialFraud  
0            1.440000             No  
1            3.674242            Yes  
2            1.429530             No  
3            1.088412            Yes  
4            0.958333             No  


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Convert 'Yes'/'No' to 1 (Fraud) and 0 (Not Fraud)
df_final['PotentialFraud'] = df_final['PotentialFraud'].map({'Yes': 1, 'No': 0})

# 2. Define Features (X) and Target (y)
# We drop 'Provider' because an ID number doesn't help predict fraud
X = df_final.drop(['Provider', 'PotentialFraud'], axis=1)
y = df_final['PotentialFraud']

# 3. Split the data (80% for training, 20% for testing)
# 'stratify=y' ensures both sets have the same ratio of fraud to non-fraud
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Initialize and Train the Random Forest Classifier
print("Training Random Forest Model...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 5. Make predictions and evaluate
y_pred_rf = rf_model.predict(X_test)
print("\n--- Random Forest Results ---")
print(classification_report(y_test, y_pred_rf))

Training Random Forest Model...

--- Random Forest Results ---
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       981
           1       0.72      0.57      0.64       101

    accuracy                           0.94      1082
   macro avg       0.84      0.78      0.80      1082
weighted avg       0.94      0.94      0.94      1082



In [5]:
from sklearn.ensemble import GradientBoostingClassifier

# 1. Initialize and Train the Gradient Boosting Classifier
print("Training Gradient Boosting Model...")
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb_model.fit(X_train, y_train)

# 2. Make predictions and evaluate
y_pred_gb = gb_model.predict(X_test)
print("\n--- Gradient Boosting Results ---")
print(classification_report(y_test, y_pred_gb))

Training Gradient Boosting Model...

--- Gradient Boosting Results ---
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       981
           1       0.77      0.56      0.65       101

    accuracy                           0.94      1082
   macro avg       0.86      0.77      0.81      1082
weighted avg       0.94      0.94      0.94      1082

